# 특허 기술이전 수요예측 — Colab GPU 실행 (최종)

맥북 잠자기/뚜껑 걱정 없이 **무료 GPU**로 돌립니다.

## ⚠️ 가장 먼저 (필수)

상단 메뉴 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU ▸ 저장** 으로 GPU를 켜세요.
그 다음 아래 셀을 **순서대로** 실행하면 됩니다. (런타임이 끊겨 다시 시작해야 하면, GPU 설정 후 셀 1부터 다시 돌리면 됩니다 — 셀 1은 몇 번 돌려도 안전)


## 셀 1 — 셋업 (GPU확인 + 코드 + 라이브러리 + 데이터, 한 번에)
재활용/끊김에 강하게 통합돼 있습니다. 이 한 셀만 다시 돌리면 환경이 복구됩니다.


In [ ]:
import torch, os
assert torch.cuda.is_available(), '❌ GPU 없음! 런타임▸런타임 유형 변경▸T4 GPU 설정 후 다시 실행하세요.'
if not os.path.exists('/content/repo'):
    !git clone https://github.com/joshlee614/Patent-Citation-Graph-Based-Technology-Transfer-Demand-Prediction-.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q torch_geometric statsmodels
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/kipris_data'   # ← 본인 데이터 폴더 경로 확인
EMB = os.path.join(DATA_DIR, 'patent_embeddings.pt')
import torch_geometric
print('✅ GPU:', torch.cuda.get_device_name(0), '| DATA OK:', os.path.exists(EMB))
!git log --oneline -1


## 셀 2 — 빠른 검증 (seeds 3, ~25분)

먼저 3 seeds로 정상 동작 확인. 결과는 **드라이브에 직접 저장**(세션 끊겨도 안전, 단 완료돼야 기록).


In [ ]:
!python run_ipm_experiment.py --mode full --seeds 3 --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 200 --artifact_dir /content/drive/MyDrive/kipris_data/ipm_check_s3


## 셀 3 — 최종 10-seed (~70분)

`--demand_sample`은 E19(거의 무의미한 베이스라인) 한 줄에만 영향. 메인 23모델은 항상 전체 테스트셋.


In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" \
  --demand_sample 200 --artifact_dir /content/drive/MyDrive/kipris_data/ipm_results_final


## 셀 4 — 결과 확인 (절대경로라 cwd 무관)


In [ ]:
print(open('/content/drive/MyDrive/kipris_data/ipm_results_final/run_ipm_results.md').read())


---
### 끊김 방지
- **탭 활성 유지 + 맥 안 자게**(뚜껑 열어두기 / 맥 터미널 `caffeinate -d`). 70분간 가끔 클릭.
- 끊기면 GPU 설정 확인 후 **셀 1 → 셀 3** 다시.
- GPU가 계속 안 잡히면 무료 quota 소진일 수 있음(~12~24h 후 재시도 또는 다른 계정).
